In [1]:
import pandas as pd

In [2]:
cases = pd.read_csv("../2일차_실습//02-01_센서_감지가능성_사례카드.csv")

In [3]:
cases

,case_id,현상,태그,주기_sec,분해능,범위_min,범위_max,변화폭,변화방향,기준값,지속_sec,예상_검토
0,A1,베어링 초기 손상으로 진동 속도가 기준보다 0.05 mm/s 증가하고 10분 지속,MTR01_VIB_RMS_H,60,0.1,0,25,0.05,상승,2.0,600,변화폭이 분해능보다 작음
1,A2,전류가 정상보다 20 A 높아지는 과전류가 5초만 지속,MTR01_CURRENT,60,0.5,0,120,20.00,상승,60.0,5,현상 지속 시간이 저장 주기보다 짧음
2,A3,모터 권선 온도가 정상 80 C에서 95 C로 상승해 30분 유지,MTR01_TEMP,60,0.5,0,180,15.00,상승,80.0,1800,지속·분해능·범위 요건을 만족
3,B1,밸브 급폐로 압력이 230 bar까지 오르는 2초 서지,HYD01_PRESS_IN,60,1.0,0,210,78.00,상승,152.0,2,주기보다 짧고 측정 상한을 초과
4,B2,유압 누유로 압력이 152 bar에서 147 bar까지 30분에 걸쳐 하강,HYD01_PRESS_IN,60,1.0,0,210,5.00,하강,152.0,1800,총 변화는 분해능보다 크고 범위 안; 작은 단계는 누적 추세로 검토
5,B3,"가열로 1존만 기준보다 6 C 높고 20분 지속, 2존은 안정",FUR01_TEMP_Z1,300,0.5,0,1300,6.00,상승,1180.0,1200,1존 변화는 감지 후보; 구역 편차를 보려면 Z2와 동시 비교 필요


In [4]:
def judge(row):
    duration_ok = row["지속_sec"] >= row["주기_sec"]
    resolution_ok = row["변화폭"] > row["분해능"]
    start = row["기준값"]
    end = start - row["변화폭"] if row["변화방향"] == "하강" else start + row["변화폭"]
    range_ok = (
        row["범위_min"] <= start <= row["범위_max"]
        and row["범위_min"] <= end <= row["범위_max"]
    )
    passed = duration_ok and resolution_ok and range_ok
    reasons = []
    if not duration_ok:
        reasons.append("현상이 저장 주기보다 짧음")
    if not resolution_ok:
        reasons.append("변화 폭이 분해능 이하")
    if not range_ok:
        reasons.append("측정 범위 초과 또는 상한 포화 가능")
    if passed:
        reasons.append("세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인")
    return duration_ok, resolution_ok, range_ok, "조건부 감지 가능" if passed else "감지 어려움", "; ".join(reasons)


In [5]:
results = cases.apply(judge, axis=1, result_type="expand")

In [6]:
results

,0,1,2,3,4
0,True,False,True,감지 어려움,변화 폭이 분해능 이하
1,False,True,True,감지 어려움,현상이 저장 주기보다 짧음
2,True,True,True,조건부 감지 가능,세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인
3,False,True,False,감지 어려움,현상이 저장 주기보다 짧음; 측정 범위 초과 또는 상한 포화 가능
4,True,True,True,조건부 감지 가능,세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인
5,True,True,True,조건부 감지 가능,세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인


In [7]:
results.columns = ["주기조건", "분해능조건", "범위조건", "1차판정", "근거"]

In [8]:
print(pd.concat([cases[["case_id", "현상", "태그"]], results], axis=1).to_string(index=False))
print(
    "\n주의: 세 조건은 데이터 수집 가능성을 검토하는 간이 체크입니다. "
    "실제 검출 성능은 노이즈, 센서 오차, 설치, 운전 조건, 필터링·저장 로직을 "
    "확인해야 하며 이 판정은 고장 진단이나 안전 판단을 대체하지 않습니다."
)

case_id                                            현상              태그  주기조건  분해능조건  범위조건      1차판정                                   근거
     A1 베어링 초기 손상으로 진동 속도가 기준보다 0.05 mm/s 증가하고 10분 지속 MTR01_VIB_RMS_H  True  False  True    감지 어려움                         변화 폭이 분해능 이하
     A2                전류가 정상보다 20 A 높아지는 과전류가 5초만 지속   MTR01_CURRENT False   True  True    감지 어려움                       현상이 저장 주기보다 짧음
     A3          모터 권선 온도가 정상 80 C에서 95 C로 상승해 30분 유지      MTR01_TEMP  True   True  True 조건부 감지 가능 세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인
     B1                밸브 급폐로 압력이 230 bar까지 오르는 2초 서지  HYD01_PRESS_IN False   True False    감지 어려움 현상이 저장 주기보다 짧음; 측정 범위 초과 또는 상한 포화 가능
     B2     유압 누유로 압력이 152 bar에서 147 bar까지 30분에 걸쳐 하강  HYD01_PRESS_IN  True   True  True 조건부 감지 가능 세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인
     B3            가열로 1존만 기준보다 6 C 높고 20분 지속, 2존은 안정   FUR01_TEMP_Z1  True   True  True 조건부 감지 가능 세 가지 1차 조건은 통과; 운전 조건·잡음·센서 상태 추가 확인

주의: 세 조건은 데이터 수집 가능성을 검토하는 간이 체크입니다. 실제 검출 성능은 